# Notebook 3 – Data Quality Assessment

Data Quality Assessment is the process of checking whether a dataset is complete, accurate, consistent, and reliable before using it for data analysis or machine learning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 104, 106, 107],
    "Name": ["Ravi", "Priya", "Amit", "Sneha", "Sneha", "Kiran", "Ravi"],
    "Age": [25, 30, np.nan, 28, 28, np.nan, 25],
    "City": ["Hyderabad", "Chennai", "Delhi", np.nan,
             np.nan, "Mumbai", "Hyderabad"],
    "Salary": [30000, 45000, 50000, 40000, 40000, np.nan, 30000]
})

df

,Customer_ID,Name,Age,City,Salary
0,101,Ravi,25.0,Hyderabad,30000.0
1,102,Priya,30.0,Chennai,45000.0
2,103,Amit,NaN,Delhi,50000.0
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0
5,106,Kiran,NaN,Mumbai,NaN
6,107,Ravi,25.0,Hyderabad,30000.0


### Number of missing values

The number of missing values tells us how many entries are unavailable in each column or throughout the entire dataset


In [2]:

missing_counts = df.isnull().sum()

print(missing_counts)

Customer_ID    0
Name           0
Age            2
City           2
Salary         1
dtype: int64


In [3]:
total_missing = df.isnull().sum().sum()          #To calculate total number of missing values

print("Total missing values:", total_missing)

Total missing values: 5


In [ ]:
Explanation:

- isnull() identifies missing values and returns True or False
- sum() counts the True values in each column
- The second sum() adds the missing counts across all columns

### Percentage of missing values

Concept: Missing value percentage shows how much of a column's data is unavailable. It helps us prioritize columns that need cleaning.

Formula:


$$
\text{Missing Percentage}
=
\frac{\text{Number of Missing Values}}
{\text{Total Number of Rows}}
\times 100
$$

In [4]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage.round(2))

Customer_ID     0.00
Name            0.00
Age            28.57
City           28.57
Salary         14.29
dtype: float64


### Missing values by column

This analysis combines the count and percentage of missing values into one table, making it easier to understand the data quality of each column

In [5]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(2),
    "Available_Count": df.notnull().sum()
})

missing_summary

,Missing_Count,Missing_Percentage,Available_Count
Customer_ID,0,0.00,7
Name,0,0.00,7
Age,2,28.57,5
City,2,28.57,5
Salary,1,14.29,6


In [6]:
missing_summary[missing_summary["Missing_Count"] > 0]       #To display only columns that contain missing values

,Missing_Count,Missing_Percentage,Available_Count
Age,2,28.57,5
City,2,28.57,5
Salary,1,14.29,6


### Missing value patterns

Missing value patterns show which columns have missing values in the same rows. This can help us understand whether missingness is isolated or occurs together

In [7]:
df[df.isnull().any(axis=1)]      #Identify rows containing missing values

,Customer_ID,Name,Age,City,Salary
2,103,Amit,NaN,Delhi,50000.0
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0
5,106,Kiran,NaN,Mumbai,NaN


In [8]:
df.isnull()         #Display a missing value pattern matrix

,Customer_ID,Name,Age,City,Salary
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,True,False,False
3,False,False,False,True,False
4,False,False,False,True,False
5,False,False,True,False,True
6,False,False,False,False,False


## Duplicate Records

### Identify duplicates

Duplicate records are rows that repeat an existing record or appear to describe the same real-world entity more than once

Pandas provides duplicated() to identify repeated rows. By default, it marks subsequent occurrences as duplicates and keeps the first occurrence unmarked

In [10]:
                                            # Identify duplicate rows across all columns
duplicate_mask = df.duplicated()

print(duplicate_mask)

0    False
1    False
2    False
3    False
4     True
5    False
6    False
dtype: bool


In [11]:
df[df.duplicated()]         #To Display Duplicate  rows

,Customer_ID,Name,Age,City,Salary
4,104,Sneha,28.0,NaN,40000.0


In [ ]:
Explanation:
Row 4 repeats row 3 across all columns, including the missing value in City. Therefore, Pandas identifies row 4 as a duplicate

### Count duplicates

Counting duplicates helps us understand how many repeated records are present.

In [12]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 1


In [13]:
duplicate_groups = df[df.duplicated(keep=False)]    #To count every occurrence in groups of repeated rows, including the first occurrence

duplicate_groups

,Customer_ID,Name,Age,City,Salary
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0


### Identify completely duplicated rows

Completely duplicated rows have the same values across every column. They may result from repeated file imports, duplicate submissions, or data integration problems

In [14]:

complete_duplicate_mask = df.duplicated(keep=False)
completely_duplicated_rows = df[complete_duplicate_mask]

completely_duplicated_rows

,Customer_ID,Name,Age,City,Salary
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0


In [ ]:
Explanation: Both rows are displayed because keep=False flags all occurrences, not just the later one

In [15]:
print("Rows belonging to duplicate groups:",   #To count the number of rows belonging to completely duplicated groups
      complete_duplicate_mask.sum())

Rows belonging to duplicate groups: 2


### Identify potential duplicate records

Potential duplicates are records that may refer to the same person, customer, product, or transaction, even when their complete rows are not identical.

For example, two records may have the same customer ID but different salaries, or the same name and city but different IDs. These cases require investigation rather than automatic deletion

In [17]:
df[df.duplicated(subset=["Customer_ID"], keep=False)]   # Check duplicate customer IDs

,Customer_ID,Name,Age,City,Salary
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0


In [18]:
df[df.duplicated(subset=["Name", "City"], keep=False)] #Check potential duplicates using multiple columns

,Customer_ID,Name,Age,City,Salary
0,101,Ravi,25.0,Hyderabad,30000.0
3,104,Sneha,28.0,NaN,40000.0
4,104,Sneha,28.0,NaN,40000.0
6,107,Ravi,25.0,Hyderabad,30000.0


In [19]:
potential_duplicate_count = df.duplicated(         #Count potential duplicates by customer ID
    subset=["Customer_ID"],
    keep="first"
).sum()

print("Repeated Customer_ID records:", potential_duplicate_count)

Repeated Customer_ID records: 1


## Invalid Values

Invalid values are data entries that do not follow the expected rules, valid ranges, formats, or business requirements of a dataset

For example:

- A negative age (-5) is inappropriate
- An age of 250 is likely impossible for a human customer
- A category such as "Hyderbad" may be misspelled
- A date such as "2026-15-40" is invalid
- A number stored as text may have an incorrect data type
- An empty string ("") may represent missing data

The objective is to identify these values before performing data analysis or building a machine learning model

### Negative values where inappropriate

CNegative values are invalid when the business meaning of a column does not allow values below zero. Examples include age, salary, quantity sold, and distance.

A negative bank balance, however, can be valid, so always consider the meaning of the column

Formula:

$$
x < 0 \Rightarrow \text{Potentially Invalid}
$$

In [23]:
import pandas as pd                  #Dataset
import numpy as np

df = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Age": [25, -5, 250, 30, 28, 35, 40, 27],
    "Salary": [30000, 45000, -1000, 50000, 40000, 35000, 60000, 32000],
    "City": [
        "Hyderabad", "Chennai", "Delhi", "Hydrabad",
        "", "Mumbai", "Unknown", None
    ],
    "Join_Date": [
        "2025-01-15", "2025-02-20", "2026-15-40",
        "2025-04-10", "2025-05-12", "not-a-date",
        "2025-07-18", "2025-08-25"
    ],
    "Experience": ["2", "5", "three", "4", "1", "0", "7", "3"]
})

df

,Customer_ID,Age,Salary,City,Join_Date,Experience
0,101,25,30000,Hyderabad,2025-01-15,2
1,102,-5,45000,Chennai,2025-02-20,5
2,103,250,-1000,Delhi,2026-15-40,three
3,104,30,50000,Hydrabad,2025-04-10,4
4,105,28,40000,,2025-05-12,1
5,106,35,35000,Mumbai,not-a-date,0
6,107,40,60000,Unknown,2025-07-18,7
7,108,27,32000,NaN,2025-08-25,3


In [24]:
df[df["Age"] < 0]

,Customer_ID,Age,Salary,City,Join_Date,Experience
1,102,-5,45000,Chennai,2025-02-20,5


In [25]:
df[df["Salary"] < 0]  #Check negative salaries

,Customer_ID,Age,Salary,City,Join_Date,Experience
2,103,250,-1000,Delhi,2026-15-40,three


In [26]:
numeric_columns = ["Age", "Salary"]         #To check numerical columns

for column in numeric_columns:
    print(f"\nNegative values in {column}:")
    print(df[df[column] < 0][["Customer_ID", column]])


Negative values in Age:
   Customer_ID  Age
1          102   -5

Negative values in Salary:
   Customer_ID  Salary
2          103   -1000


### Impossible values

Impossible values fall outside a realistic or logically permitted range. These differ from negative values because a value can be positive but still invalid.

For example, an age of 250 is unrealistic for a human custome

Formula:

$$
x < L \quad \text{or} \quad x > U
$$

Where:

- \(x\) = Observed value
- \(L\) = Minimum acceptable value
- \(U\) = Maximum acceptable value

In [27]:

min_age = 0
max_age = 120

invalid_age_mask = (
    (df["Age"] < min_age) |
    (df["Age"] > max_age)
)

df.loc[invalid_age_mask, ["Customer_ID", "Age"]]

,Customer_ID,Age
1,102,-5
2,103,250


In [28]:
salary_threshold = 1_000_000

df[df["Salary"] > salary_threshold]

,Customer_ID,Age,Salary,City,Join_Date,Experience


### Incorrect categories

Incorrect categories are values that do not match the allowed category labels. They may occur because of spelling mistakes, inconsistent capitalization, extra spaces, or unexpected labels

For example, "Hydrabad" may be a misspelling of "Hyderabad"

Formula:

$$
x \notin C \Rightarrow \text{Invalid Category}
$$

Where (C) is the set of allowed categories

In [29]:

valid_cities = [
    "Hyderabad",
    "Chennai",
    "Delhi",
    "Mumbai"
]

invalid_categories = df[
    df["City"].notna() &
    ~df["City"].isin(valid_cities) &
    df["City"].ne("")
]

invalid_categories[["Customer_ID", "City"]]

,Customer_ID,City
3,104,Hydrabad
6,107,Unknown


In [30]:
print(df["City"].unique()) #To check unique categories

<ArrowStringArray>
['Hyderabad', 'Chennai', 'Delhi', 'Hydrabad', '', 'Mumbai', 'Unknown', nan]
Length: 8, dtype: str


### Invalid dates

Invalid dates are date values that cannot be interpreted as real calendar dates or do not follow the expected date rules

Examples include "2026-15-40", "not-a-date", or a future date in a dataset that should only contain past joining dates

Formula:

I(\text{Date Cannot Be Parsed})
$$

Where (I) is an indicator function

In [31]:

parsed_dates = pd.to_datetime(
    df["Join_Date"],
    errors="coerce"
)

invalid_date_mask = (
    df["Join_Date"].notna() &
    parsed_dates.isna()
)

df.loc[invalid_date_mask, ["Customer_ID", "Join_Date"]]

,Customer_ID,Join_Date
2,103,2026-15-40
5,106,not-a-date


### Incorrect data types

Incorrect data types occur when a column's stored type does not match its intended meaning.

Examples:

- Age stored as text instead of a number
- Salary stored as text instead of a numeric value
- Joining dates stored as strings instead of datetime values

Formula:

$$
\text{Actual Type} \ne \text{Expected Type}
$$

In [32]:
print(df.dtypes)

Customer_ID    int64
Age            int64
Salary         int64
City             str
Join_Date        str
Experience       str
dtype: object


## Unexpected values

Unexpected values are entries that are unusual or violate a business rule, even if their data type and format are correct

Examples include:

- An unsupported customer status
- An unusually high salary
- A quantity that exceeds a permitted limit
- A value outside the documented list of acceptable options

Formula:


$$
x \notin V \Rightarrow \text{Unexpected Value}
$$

Where \(V\) is the set of expected values defined by the business rules

In [39]:
import pandas as pd
import numpy as np

experience = pd.to_numeric(df["Experience"], errors="coerce")

min_experience = 0
max_experience = 50

unexpected_mask = (
    (experience < min_experience) |
    (experience > max_experience)
)

df.loc[unexpected_mask, ["Customer_ID", "Experience"]]

,Customer_ID,Experience


In [37]:
category_counts = df["City"].value_counts(   #Detect rare categories
    dropna=False
)

print(category_counts)

City
Hyderabad    1
Chennai      1
Delhi        1
Hydrabad     1
             1
Mumbai       1
Unknown      1
NaN          1
Name: count, dtype: int64


### Empty strings

An empty string ("") contains no characters. Pandas does not normally treat it as a missing value, so isnull() will not detect it

Formula:

$$
x = ""
$$

In [40]:

empty_string_counts = (
    df.select_dtypes(include=["object", "string"])
      .eq("")
      .sum()
)

print(empty_string_counts)

City          1
Join_Date     0
Experience    0
dtype: int64


In [41]:
empty_city_mask = df["City"].eq("")       #Identify rows containing empty strings

df.loc[empty_city_mask]

,Customer_ID,Age,Salary,City,Join_Date,Experience
4,105,28,40000,,2025-05-12,1


In [42]:
whitespace_mask = (      #Detect strings containing only whitespace
    df["City"].notna() &
    df["City"].str.strip().eq("")
)

df.loc[whitespace_mask]

,Customer_ID,Age,Salary,City,Join_Date,Experience
4,105,28,40000,,2025-05-12,1


### Null-like values

Null-like values are text placeholders that represent missing information, such as "NA", "N/A", "null", "None", or "missing"

These are strings, so Pandas may treat them as ordinary text rather than actual missing values

Formula:

$$
x \in S
\Rightarrow
\text{Potential Missing Value}
$$

Where (S) is the set of recognized null-like values, such as "NA", "null", and "None"

In [43]:

null_like_values = [
    "NA",
    "N/A",
    "null",
    "None",
    "missing",
    "unknown"
]

example = pd.Series([
    "Hyderabad",
    "NA",
    "null",
    "Chennai",
    "None",
    ""
])

null_like_mask = (
    example.astype("string")
           .str.strip()
           .str.lower()
           .isin([value.lower() for value in null_like_values])
)

print(example[null_like_mask])

1      NA
2    null
4    None
dtype: str
